# Qwen 2.5 14B — Few-Shot v4 (Soft Few-Shot Interpretativo)

**Hipótesis de trabajo**: v2/v3 son demasiado *imperativos* ("mandatory checklist", "Step 1", "do not invent"). El modelo sigue órdenes en vez de interpretar. v1 falló por exceso de ejemplos rígidos (efecto loro). El punto medio:

1. **Tono interpretativo**: el algoritmo deductivo se presenta como *marco de referencia*, no como comando.
2. **Tres casos de calibración** del propio corpus UPV — narrativa breve por caso, sin formato Q→A — que muestran el *criterio* sin imponer la *plantilla*.
3. **Anti-parrot suave**: "these illustrate the kind of judgment expected. Your abstract is different — do not reuse their wording."

**Casos de calibración** (extraídos del subset evaluado, marcados para exclusión):
- **A** `b9e1bf330baf` — PB4 inequívoco: bacterias anammox + ciclo del N en aguas residuales (medición biofísica con flux analysis).
- **B** `9153f4dcf3d6` — None: verificación de modelo WRF (paper metodológico que menciona clima como contexto pero no lo mide).
- **C** `f21bc832abc6` — PB7: respuesta de mamíferos a la huella humana (biodiversidad cuantificada con porcentajes de especies).

Estos 3 doc_ids se **excluyen** de la inferencia y de las métricas. El subset evaluado pasa de 150 → 147.

In [16]:
import pandas as pd
import requests
import json
import time
import os
from pathlib import Path

## 1. Carga de datos y reglas PB

In [17]:
ROOT_DIR = Path.cwd().resolve().parents[2]
LLM_DIR = Path.cwd().resolve().parents[0]
OUTPUTS_DIR = LLM_DIR / 'outputs' / 'inferences'
GROUND_TRUTH_DIR = LLM_DIR / 'outputs' / 'ground_truth'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

EXAMPLE_DOC_IDS = {'b9e1bf330baf', '9153f4dcf3d6', 'f21bc832abc6'}

df_corpus = pd.read_csv(ROOT_DIR / 'data' / 'corpus' / 'master_corpus_mixto_1000_clean_enriched.csv')
df_pbs = pd.read_csv(ROOT_DIR / 'corpus_PB' / 'data' / 'pb_reference.csv')
df_human = pd.read_csv(GROUND_TRUTH_DIR / 'validacion_real.csv', sep=';', encoding='utf-8')
ids_validados = df_human['doc_id'].dropna().astype(str).unique().tolist()

df_sample = df_corpus[df_corpus['doc_id'].isin(ids_validados)].copy()
df_sample = df_sample[~df_sample['doc_id'].isin(EXAMPLE_DOC_IDS)].copy()
print(f'Papers a evaluar (excluyendo los 3 ejemplos del prompt): {len(df_sample)}')

pb_rules = ''
for _, row in df_pbs.iterrows():
    pb_rules += f"- PB Code: {row['pb_code']} ({row['pb_name']})\n"
    pb_rules += f"  * Core Definition: {row['short_definition']}\n"
    pb_rules += f"  * UPV Context: Look for terms like: {row['applied_keywords_upv']}\n"
    pb_rules += f"  * ACTIVATION LOGIC: {row['activation_logic']}\n"
    pb_rules += f"  * EXCLUSION RULE (CRITICAL): {row['exclusion_notes']}\n\n"

Papers a evaluar (excluyendo los 3 ejemplos del prompt): 146


## 2. Prompt v4 — marco interpretativo + 3 calibration cases

In [ ]:
OLLAMA_URL = 'http://localhost:11434/api/generate'
OLLAMA_TAGS_URL = 'http://localhost:11434/api/tags'
MODEL_NAME = 'qwen2.5:14b'

import time
import json
import requests

def preflight_check(model_name):
    """Verifica que Ollama responde y que el modelo existe. Lanza excepción si no."""
    try:
        r = requests.get(OLLAMA_TAGS_URL, timeout=5)
        r.raise_for_status()
    except Exception as e:
        raise RuntimeError(f'Ollama no responde en {OLLAMA_TAGS_URL}: {e}')
    available = [m.get('name') for m in r.json().get('models', [])]
    if model_name not in available:
        raise RuntimeError(
            f"Modelo '{model_name}' no está cargado en Ollama. "
            f"Modelos disponibles: {available}. "
            f"Descárgalo con: ollama pull {model_name}"
        )
    print(f"OK: Ollama responde y '{model_name}' está disponible.")

def classify_abstract_v4(abstract_text):
    # =========================================================================
    # PROMPT MAESTRO: Enfoque Basado en Principios (Principle-Driven) + Etiquetas XML
    # Diseñado para curar Falsos Negativos y el Positivity Bias del contexto
    # =========================================================================
    
    prompt = f"""<system_role>
You are an expert scientific evaluator analyzing research abstracts from the Universitat Politècnica de València (UPV) against the Planetary Boundaries (PBs) framework. Your judgment matters more than rigid rule-following. Use the provided framework as a reasoning aid, not a script.
</system_role>

<task>
Your goal is to identify the Planetary Boundaries that the research actually MEASURES or MODELS, strictly separating real biophysical analysis from mere background or motivational context.
</task>

<instructions>
To achieve this, follow these analytical steps:
1. Identify the *operational object*: Determine the exact variable being measured, modelled, or experimentally manipulated. Quantification, methodology, comparisons, or model outputs around a variable indicate it is the focus. Mere mention without numbers or methods is just background.
2. Match with PB: Determine which PB's Core Definition/Activation Logic best matches that operational object. The *primary* PB is the one being analytically resolved, not the one mentioned in the introduction.
3. Apply EXCLUSION RULES: If the abstract is purely social, legal, governance, education, philosophy, or pure-software theory, and the biophysical terms are only motivational, the verdict must be "None".
4. Check common confusions:
   - Aerosols / PM / AOD belong to PB9, not PB1.
   - "Climate" or "warming" alone is not enough for PB1. There must be an explicit climate driver, scenario, or ecological climate impact being studied.
   - Biodiversity / ecosystem responses are a legitimate primary candidate for PB7.
   - Nutrients (N, P) point to PB4.
   - Water quantity/quality point to PB5. (Pick whichever variable the paper actually resolves).
5. Think step-by-step: Analyze the text and formulate a very high-level summary of your reasoning before outputting the final JSON. Keep only the most direct steps leading to your conclusion.
</instructions>

<reference_framework>
{pb_rules}
</reference_framework>

<calibration_cases>
These cases share one principle: *the primary PB is whatever variable the paper quantifies or models*, regardless of the introduction's rhetorical framing. Use them ONLY to calibrate your judgment, NOT to mimic their wording.

- Case A: A paper studies the central carbon metabolism of anammox bacteria using 13C isotope tracing. The abstract reports specific metabolic pathways and quantified findings. Focus: Nitrogen biogeochemistry (measured, not just mentioned). Primary PB: PB4. Unambiguous.
- Case B: A paper verifies the WRF forecast model comparing initial conditions against radar data. It mentions "changing climate" in the opening. Focus: Forecast-model accuracy. Climate is motivational backdrop, not a measured variable. Primary PB: None.
- Case C: A paper quantifies detection data for 24 mammal species in response to human footprint. Focus: Ecological response of biodiversity to human pressure (measured with statistics). Primary PB: PB7. PB6 (land-system change) is just a contextual driver.
</calibration_cases>

<input_data>
<abstract>
{abstract_text}
</abstract>
</input_data>

<constraints>
- DO NOT reuse phrases from the calibration cases. Your abstract deserves a fresh reading.
- Write your own reasoning, citing the abstract's own terms.
- DO NOT output any markdown formatting (like ```json).
- DO NOT include any conversational text, greetings, or explanations outside the JSON structure.
</constraints>

<output_format>
Return ONLY valid JSON in the exact structure below:
{{
    "reasoning_process": "A very high-level summary of your reasoning (2-3 sentences). State exactly what is measured/modelled, why you chose this PB, and why it is not background context. Omit intermediate details.",
    "primary_pb": {{"pb_code": "PBX", "confidence": "High/Medium/Low"}} or null,
    "secondary_pbs": [{{"pb_code": "PBY", "confidence": "High/Medium/Low"}}],
    "rejected_pbs": ["PBZ", "PBW"]
}}
</output_format>"""

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "format": "json",
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 0.9
        }
    }

    start_time = time.time()
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=180)
        response.raise_for_status()
        result_json = response.json()
        return result_json.get("response", "{}"), time.time() - start_time, None
    except Exception as e:
        # Devuelve también el error como tercer valor para que el bucle lo imprima
        err_msg = f'{type(e).__name__}: {str(e)[:200]}'
        return json.dumps({"error": err_msg, "primary_pb": None, "secondary_pbs": [], "rejected_pbs": []}), time.time() - start_time, err_msg

## 3. Parser estándar

In [19]:
def parse_llm_output(raw_text):
    try:
        si, ei = raw_text.find('{'), raw_text.rfind('}')
        if si == -1 or ei == -1:
            raise ValueError('No JSON structure found.')
        data = json.loads(raw_text[si:ei + 1])

        reasoning = data.get('reasoning_process', '')
        prim = data.get('primary_pb')
        if isinstance(prim, dict):
            prim_code = prim.get('pb_code', 'None')
            prim_conf = prim.get('confidence', 'Unknown')
        else:
            prim_code, prim_conf = 'None', 'N/A'
        sec = data.get('secondary_pbs', []) or []
        sec_codes = ', '.join([s.get('pb_code', '') for s in sec if isinstance(s, dict)]) or 'None'
        rej = data.get('rejected_pbs', []) or []
        rej_codes = ', '.join(rej) if rej else 'None'
        return reasoning, prim_code, prim_conf, sec_codes, rej_codes
    except Exception as e:
        return f'Error: {e}', 'Error', 'Error', 'Error', 'Error'

## 4. Bucle de evaluación

In [ ]:
output_filename = OUTPUTS_DIR / 'qwen2.5_14b_fewshot_v4_principle.csv'
if os.path.exists(output_filename):
    os.remove(output_filename)

# Pre-flight: aborta limpio si Ollama no responde o el modelo no está cargado.
preflight_check(MODEL_NAME)

total = len(df_sample)
print(f'FEWSHOT v4 (principle-driven + XML + 3 calibration cases) | modelo: {MODEL_NAME}')
print(f'Excluidos del eval: {sorted(EXAMPLE_DOC_IDS)}')
print('-' * 70)

err_count = 0
for i, (_, row) in enumerate(df_sample.iterrows(), start=1):
    print(f"[{i}/{total}] {row['doc_id']}...", end=' ', flush=True)
    raw, t, err = classify_abstract_v4(row['clean_abstract'])
    if err:
        err_count += 1
        print(f'[{t:.2f}s] FALLO -> {err}')
        if err_count >= 3:
            print('\n>>> 3 fallos consecutivos, abortando para que diagnostiques. <<<')
            break
    else:
        err_count = 0  # reset en cuanto vuelva a funcionar
    reasoning, p, pc, s, r = parse_llm_output(raw)
    if not err:
        print(f'[{t:.1f}s] Pri: {p} | Sec: {s} | Rej: {r}')
    pd.DataFrame([{
        'doc_id': row['doc_id'],
        'llm_primary_pb': p,
        'llm_primary_conf': pc,
        'llm_secondary_pbs': s,
        'llm_rejected_pbs': r,
        'llm_reasoning': reasoning,
        'inference_time_sec': t,
    }]).to_csv(output_filename, mode='a', header=not output_filename.exists(), index=False)

print('=' * 70)
print(f'OK. CSV: {output_filename}')